In [1]:
import os

# 1. Manually set your credentials
os.environ['KAGGLE_USERNAME'] = "mlpython369" # Put your username inside the quotes
os.environ['KAGGLE_KEY'] = "KGAT_291fb487d0b051838409c7d4433ee547"           # Paste that long key here

# 2. Check if it works by trying to download
!kaggle datasets download -d puneet6060/intel-image-classification

Dataset URL: https://www.kaggle.com/datasets/puneet6060/intel-image-classification
License(s): copyright-authors
 77% 268M/346M [00:00<00:00, 649MB/s] 
100% 346M/346M [00:00<00:00, 619MB/s]


In [2]:
#unzipping the data

!unzip -q intel-image-classification.zip

In [3]:
import tensorflow as tf

BATCH_SIZE = 32
IMG_SIZE = (160, 160) # MobileNetV2 prefers this size

# Load training data
train_ds = tf.keras.utils.image_dataset_from_directory(
    'seg_train/seg_train',
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE
)

# Load validation data
val_ds = tf.keras.utils.image_dataset_from_directory(
    'seg_test/seg_test',
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE
)

# Optimization: Prefetching (The performance "secret sauce")
#tells TensorFlow to look at your hardware in real-time and decide how many batches to prefetch to get the max speed without crashing your memory.
AUTOTUNE = tf.data.AUTOTUNE
#Normally, every time a new Epoch (one full pass through the data) starts, the computer has to open the image files from the disk all over again.
#While the GPU is "eating" (training on) Batch 1, the CPU is already "cooking" (loading and resizing) Batch 2.
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

Found 14034 files belonging to 6 classes.
Found 3000 files belonging to 6 classes.


In [4]:
# Load the pre-trained MobileNetV2
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False, # We don't want Google's 1000-class head
    weights='imagenet'
)

base_model.trainable = False #stops the model from updating the weights : Protecting Existing Knowledge + speed+resource management

# Attach our custom "Head"
model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(6, activation='softmax') # Our 6 scenery classes
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_160            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 6)              │         7,686 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,265,670 (8.64 MB)

 Trainable params: 7,686 (30.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [6]:
# This will likely be the fastest training you've seen yet
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 312s 711ms/step - accuracy: 0.7735 - loss: 0.6090 - val_accuracy: 0.7807 - val_loss: 0.5834
Epoch 2/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 311s 708ms/step - accuracy: 0.7930 - loss: 0.5505 - val_accuracy: 0.7887 - val_loss: 0.5647
Epoch 3/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 316s 720ms/step - accuracy: 0.8067 - loss: 0.5167 - val_accuracy: 0.7917 - val_loss: 0.5549
Epoch 4/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 318s 712ms/step - accuracy: 0.8164 - loss: 0.4928 - val_accuracy: 0.7937 - val_loss: 0.5493
Epoch 5/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 316s 720ms/step - accuracy: 0.8250 - loss: 0.4743 - val_accuracy: 0.7980 - val_loss: 0.5462
Epoch 6/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 323s 723ms/step - accuracy: 0.8270 - loss: 0.4593 - val_accuracy: 0.8000 - val_loss: 0.5447
Epoch 7/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 313s 713ms/step - accuracy: 0.8319 - loss: 0.4466 - val_accuracy: 0.8007 - val_loss: 0.5444
Epoch 8/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 313s 713ms/step - accuracy: 0.8375 -

In [7]:
# Save the model to the Colab 'content' folder
model.save('final_model.keras')
print("Model saved successfully!")

Model saved successfully!


In [8]:
from google.colab import files

# This will trigger a browser download
files.download('final_model.keras')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>